In [ ]:
!pip install --upgrade osmnx


In [ ]:
!pip install ipywidgets folium


In [2]:
import folium

# Example: create a map
salzburg_map = folium.Map(location=[47.8, 13.04], zoom_start=13)

# Save to HTML
salzburg_map.save("salzburg_map.html")

# Show inline
salzburg_map


In [8]:
import folium
import osmnx as ox
import pandas as pd  # needed for concat

# Create base map
salzburg_map = folium.Map(location=[47.8, 13.04], zoom_start=13)

# Define the place
place_name = "Salzburg, Austria"

# Define specific POI types for each tag
tags_to_check = {
    "leisure": ["park", "garden", "playground"],
    "tourism": ["museum", "attraction", "viewpoint"],
    "amenity": ["cafe", "restaurant", "hospital", "school"]
}

# Initialize empty list to collect GeoDataFrames
gdf_list = []

# Loop through each tag and type
for tag, values in tags_to_check.items():
    for value in values:
        gdf = ox.geometries_from_place(place_name, tags={tag: value})
        gdf_list.append(gdf)

# Concatenate all GeoDataFrames into one
gdf_all = pd.concat(gdf_list, ignore_index=False)

# Remove duplicates if any
gdf_all = gdf_all[~gdf_all.index.duplicated(keep='first')]
print(f"Total POIs found: {len(gdf_all)}")

# Add markers for each POI
for _, row in gdf_all.iterrows():
    if hasattr(row.geometry, "centroid"):
        centroid = row.geometry.centroid
        name = row.get("name", "No name")
        folium.Marker(
            location=[centroid.y, centroid.x],
            popup=name,
            icon=folium.Icon(color="green", icon="info-sign")
        ).add_to(salzburg_map)

# Save and display map
salzburg_map.save("salzburg_map.html")
salzburg_map


Total POIs found: 1182


In [9]:
# Install required packages if not already installed
# !pip install osmnx==1.2 folium geopandas pandas

import os
import json
import csv
import osmnx as ox
import folium
import pandas as pd
from typing import List

# -------------------------------
# 1. Define TouristAttraction class
# -------------------------------
class TouristAttraction:
    def __init__(self, id: int, name: str, lat: float, lng: float, category: str, icon: str):
        self.id = id
        self.name = name
        self.lat = lat
        self.lng = lng
        self.category = category
        self.icon = icon
    
    def to_dict(self):
        return {
            "id": self.id,
            "name": self.name,
            "latitude": self.lat,
            "longitude": self.lng,
            "category": self.category,
            "icon": self.icon
        }
    
    def to_list(self):
        return [self.id, self.name, self.lat, self.lng, self.category, self.icon]

# -------------------------------
# 2. Helper functions
# -------------------------------
def get_valid_name(tags: dict, idx: int) -> str:
    for key in ['name', 'name:en', 'official_name']:
        if key in tags:
            return tags[key]
    return f"Attraction_{idx}"

def determine_category(tags: dict):
    if 'tourism' in tags:
        return 'tourism', 'star'
    elif 'historic' in tags:
        return 'historic', 'landmark'
    elif 'amenity' in tags:
        return 'amenity', 'info-sign'
    elif 'leisure' in tags:
        return 'leisure', 'tree'
    else:
        return 'other', 'question-sign'

# -------------------------------
# 3. Load attractions from OSM
# -------------------------------
def load_city_attractions(city_name: str, max_attractions: int = 20) -> List[TouristAttraction]:
    print(f"Loading attractions for {city_name}...")
    
    tags_to_check = {
        'tourism': ['attraction', 'museum', 'gallery', 'viewpoint', 'zoo', 'theme_park'],
        'historic': ['monument', 'castle', 'memorial', 'archaeological_site'],
        'amenity': ['place_of_worship', 'fountain', 'theatre', 'cafe', 'restaurant', 'hospital', 'school'],
        'leisure': ['park', 'garden', 'nature_reserve'],
        'building': ['church', 'cathedral', 'mosque', 'temple'],
        'shop': ['gift', 'souvenir', 'art']
    }
    
    gdf_list = []
    for tag, values in tags_to_check.items():
        for value in values:
            try:
                gdf = ox.geometries_from_place(city_name, tags={tag: value})
                gdf_list.append(gdf)
            except Exception as e:
                print(f"Warning: Failed to fetch {tag}:{value} -> {e}")
    
    gdf_all = pd.concat(gdf_list, ignore_index=False)
    gdf_all = gdf_all[~gdf_all.index.duplicated(keep='first')]
    print(f"Total POIs fetched: {len(gdf_all)}")
    
    attractions = []
    idx = 1
    for _, row in gdf_all.iterrows():
        if len(attractions) >= max_attractions:
            break
        if hasattr(row.geometry, 'centroid'):
            centroid = row.geometry.centroid
            tags_dict = dict(row.dropna())
            name = get_valid_name(tags_dict, idx)
            category, icon = determine_category(tags_dict)
            attractions.append(TouristAttraction(
                id=idx,
                name=name,
                lat=centroid.y,
                lng=centroid.x,
                category=category,
                icon=icon
            ))
            idx += 1
    
    return attractions[:max_attractions]

# -------------------------------
# 4. Save functions
# -------------------------------
def save_to_json(attractions: List[TouristAttraction], filename: str):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump([a.to_dict() for a in attractions], f, indent=2, ensure_ascii=False)

def save_to_csv(attractions: List[TouristAttraction], filename: str):
    with open(filename, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'name', 'latitude', 'longitude', 'category', 'icon'])
        for a in attractions:
            writer.writerow(a.to_list())

# -------------------------------
# 5. Main extraction & map display
# -------------------------------
def extract_and_display(city: str = "Salzburg, Austria", output_dir: str = 'output'):
    os.makedirs(output_dir, exist_ok=True)
    attractions = load_city_attractions(city)
    
    base_name = city.replace(', ', '_').replace(' ', '_').lower()
    
    json_file = f"{output_dir}/{base_name}_attractions.json"
    save_to_json(attractions, json_file)
    
    csv_file = f"{output_dir}/{base_name}_attractions.csv"
    save_to_csv(attractions, csv_file)
    
    print(f"\n✓ Extracted {len(attractions)} attractions")
    print(f"JSON saved to: {json_file}")
    print(f"CSV saved to: {csv_file}")
    
    # Folium map
    m = folium.Map(location=[47.8, 13.04], zoom_start=13)
    color_map = {'tourism':'blue','historic':'orange','amenity':'green','leisure':'purple','other':'gray'}
    
    for a in attractions:
        folium.Marker(
            location=[a.lat, a.lng],
            popup=f"{a.name} ({a.category})",
            icon=folium.Icon(color=color_map.get(a.category, 'gray'), icon='info-sign')
        ).add_to(m)
    
    return m

# -------------------------------
# 6. Run in notebook
# -------------------------------
map_salzburg = extract_and_display()
map_salzburg  # displays interactive map inline


Loading attractions for Salzburg, Austria...


C:\Users\User\miniforge3\envs\salzburg_env\lib\site-packages\osmnx\geometries.py:358: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  gdf["geometry"] = np.nan
C:\Users\User\miniforge3\envs\salzburg_env\lib\site-packages\osmnx\geometries.py:358: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geo

Total POIs fetched: 1756

✓ Extracted 20 attractions
JSON saved to: output/salzburg_austria_attractions.json
CSV saved to: output/salzburg_austria_attractions.csv
